# FastConformer-CTC PEFT Fine-Tuning (Colab)

This notebook fine-tunes a NeMo FastConformer-CTC `.nemo` model on custom NeMo manifests stored in Google Drive.

Notes:
- FastConformer ASR uses NeMo adapter-based PEFT here, which is the closest supported alternative to LoRA.
- Run the notebook top to bottom in a fresh GPU Colab runtime.
- The install cell may restart the runtime once on the first run after package changes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title 1. User Configuration

# Paths in Google Drive.
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_ROOT = f"{DRIVE_MOUNT_POINT}/MyDrive/shaswot_mate/angrezi"
DATA_ROOT = f"{DRIVE_ROOT}/asrtest"

BASE_MODEL_NEMO_PATH = f"{DRIVE_ROOT}/fastconformer.nemo"
TRAIN_MANIFEST = f"{DATA_ROOT}/train/manifest.jsonl"
VAL_MANIFEST = f"{DATA_ROOT}/val/manifest.jsonl"
TEST_MANIFEST = f"{DATA_ROOT}/test/manifest.jsonl"

# Optional resume / continue paths.
CONTINUE_FROM_FINETUNED_NEMO_PATH = None
CONTINUE_FROM_ADAPTERS_PT_PATH = None
RESUME_FROM_CHECKPOINT_PATH = None

# Output paths.
RUN_NAME = "fastconformer_ctc_small_peft"
OUTPUT_ROOT = f"{DRIVE_ROOT}/nemo_fastconformer_runs"
OVERWRITE_EXISTING_EXPORTS = True

# Dependency setup.
INSTALL_DEPENDENCIES = True
USE_COLAB_DEFAULT_NUMPY = True
NEMO_VERSION = "2.4.0"

# Reproducibility.
SEED = 42
# GPU CTC backward is not strictly deterministic in PyTorch, so warn-only mode
# keeps the run reproducible where possible without crashing training.
DETERMINISTIC = True

# Dataset settings.
SAMPLE_RATE = 16000
TRIM_SILENCE = False
MIN_DURATION = 0.5
MAX_DURATION = 20 # setting limit
NUM_WORKERS = 7  # 2 -> 7 for nice GPUs
PIN_MEMORY = True

# PEFT settings.
USE_PEFT = True
PEFT_TARGET_MODULE = "encoder"
ADAPTER_NAME = "domain_adapter"
ADAPTER_DIM = 8 #  was 32
ADAPTER_DROPOUT = 0.25 # was 0.1
ADAPTER_ACTIVATION = "swish"
ADAPTER_NORM_POSITION = "pre"


# Training settings.
MAX_EPOCHS = 12
TRAIN_BATCH_SIZE = 32
VAL_BATCH_SIZE = 64
TEST_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 64
GRAD_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 5e-2
BETAS = (0.9, 0.98)
WARMUP_RATIO = 0.1
GRAD_CLIP_VAL = 1.0
PRECISION = "bf16-mixed"
LOG_EVERY_N_STEPS = 3
CHECK_VAL_EVERY_N_EPOCH = 1
NUM_SANITY_VAL_STEPS = 0
LIMIT_TRAIN_BATCHES = 1.0
LIMIT_VAL_BATCHES = 1.0
LIMIT_TEST_BATCHES = 1.0
SUBSAMPLING_CONV_CHUNKING_FACTOR = 1
MATMUL_PRECISION = "high"

# # Checkpointing / early stopping.
# SAVE_TOP_K = 1
# PERIODIC_CHECKPOINT_EVERY_N_EPOCHS = 1
# ENABLE_EARLY_STOPPING = True
# EARLY_STOPPING_MONITOR = "val_wer"
# EARLY_STOPPING_MODE = "min"
# EARLY_STOPPING_MIN_DELTA = 0.001
# EARLY_STOPPING_PATIENCE = 2

# Augmentation for noise robustness.
ENABLE_SPEC_AUGMENT = True
SPEC_AUG_FREQ_MASKS = 2
SPEC_AUG_FREQ_WIDTH = 15
SPEC_AUG_TIME_MASKS = 6      # was 5
SPEC_AUG_TIME_WIDTH = 30     # was 25

# Checkpointing / early stopping.
SAVE_TOP_K = 1
PERIODIC_CHECKPOINT_EVERY_N_EPOCHS = 1
ENABLE_EARLY_STOPPING = True
EARLY_STOPPING_MONITOR = "val_wer"
EARLY_STOPPING_MODE = "min"
EARLY_STOPPING_MIN_DELTA = 0.0
EARLY_STOPPING_PATIENCE = 5

# Metrics.
DISABLE_NEMO_PREDICTION_LOGS = True
COMPUTE_VAL_CER_EVERY_N_EPOCHS = 3
EVAL_NUM_WORKERS = 0
LOWERCASE_TEXT_FOR_METRICS = False
COLLAPSE_WHITESPACE_FOR_METRICS = True

# Manifest validation.
FAIL_ON_PLACEHOLDER_TRANSCRIPTS = True
PLACEHOLDER_TOKENS = {"placeholder", "<placeholder>", "todo", "dummy"}


In [ ]:
#@title 2. Mount Drive and Install Dependencies

import importlib.metadata as importlib_metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run_cmd(cmd):
    print(f"[cmd] {' '.join(cmd)}")
    subprocess.run(list(cmd), check=True)


def get_dist_version(*names):
    for name in names:
        try:
            return importlib_metadata.version(name)
        except importlib_metadata.PackageNotFoundError:
            continue
    return None


def has_libraries() -> bool:
    candidates = [
        "/usr/lib/x86_64-linux-gnu/libsndfile.so.1",
        "/usr/lib/aarch64-linux-gnu/libsndfile.so.1",
        "/usr/lib/libsndfile.so.1",
    ]
    return shutil.which("ffmpeg") is not None and any(Path(path).exists() for path in candidates)


IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
RESTART_MARKER = Path("/tmp/fastconformer_peft_dependencies_ready")

# if IN_COLAB:
#     from google.colab import drive  # type: ignore

#     drive.mount(DRIVE_MOUNT_POINT, force_remount=False)
# else:
#     print("[info] Not running in Colab; skipping Google Drive mount.")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("NEMO_EXPM_VERSION", "0")
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

if INSTALL_DEPENDENCIES:
    need_apt = not has_libraries()
    need_pip = not all(
        [
            get_dist_version("nemo-toolkit", "nemo_toolkit") == NEMO_VERSION,
            get_dist_version("lightning") is not None,
            get_dist_version("matplotlib") is not None,
            get_dist_version("omegaconf") is not None,
        ]
    )

    if need_apt:
        run_cmd(["apt-get", "-qq", "update"])
        run_cmd(["apt-get", "-qq", "install", "-y", "ffmpeg", "libsndfile1"])
    else:
        print("[info] Skipping apt packages; ffmpeg and libsndfile are already present.")

    if need_pip:
        packages = [
            "Cython",
            "packaging",
            f"nemo_toolkit[asr]=={NEMO_VERSION}",
            "lightning>=2.2.0,<2.6.0",
            "matplotlib>=3.7.0",
        ]
        if not USE_COLAB_DEFAULT_NUMPY:
            packages.insert(0, "numpy==1.26.4")
        run_cmd([sys.executable, "-m", "pip", "install", "--no-input", "--upgrade", "pip"])
        run_cmd([sys.executable, "-m", "pip", "install", "--no-input", "--upgrade", *packages])

        if IN_COLAB and not RESTART_MARKER.exists():
            RESTART_MARKER.write_text("done", encoding="utf-8")
            print("[info] Dependencies changed. Restarting runtime once so the new packages load cleanly...")
            os.kill(os.getpid(), 9)
    else:
        print("[info] Skipping pip install; compatible dependencies are already present.")
else:
    print("[info] Dependency installation disabled by config.")


[info] Skipping apt packages; ffmpeg and libsndfile are already present.
[info] Skipping pip install; compatible dependencies are already present.


In [ ]:
#@title 3. Imports and Helper Functions

import csv
import json
import math
import random
import shutil
from datetime import datetime
from pathlib import Path
from statistics import mean
from typing import Dict, Iterable, List, Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import Callback, EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from omegaconf import OmegaConf, open_dict

import nemo.collections.asr as nemo_asr
from nemo.collections.common.parts import adapter_modules

if not torch.cuda.is_available():
    raise EnvironmentError(
        "CUDA GPU is required for this notebook. In Colab, go to Runtime -> Change runtime type -> GPU, then rerun from the top."
    )

DEVICE = torch.device("cuda", torch.cuda.current_device())
print(f"[info] GPU detected: {torch.cuda.get_device_name(0)}")
print(f"[info] NumPy version: {np.__version__}")
print(f"[info] CUDA version: {torch.version.cuda}")


def seed_python(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def ensure_exists(path: str | None, description: str, required: bool = True) -> None:
    if path is None:
        if required:
            raise FileNotFoundError(f"{description} is required but is None.")
        return
    if not Path(path).exists():
        raise FileNotFoundError(f"{description} not found: {path}")


def metric_to_float(value, default: float = float("nan")) -> float:
    if value is None:
        return default
    if hasattr(value, "detach"):
        return float(value.detach().cpu().item())
    try:
        return float(value)
    except Exception:
        return default


def normalize_text(text: str) -> str:
    text = text.strip()
    if COLLAPSE_WHITESPACE_FOR_METRICS:
        text = " ".join(text.split())
    if LOWERCASE_TEXT_FOR_METRICS:
        text = text.lower()
    return text


def load_manifest(manifest_path: str) -> List[Dict[str, object]]:
    entries: List[Dict[str, object]] = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_idx} in {manifest_path}: {exc}") from exc

            if "audio_filepath" not in item:
                raise KeyError(f"Missing 'audio_filepath' on line {line_idx} in {manifest_path}")
            if "text" not in item:
                raise KeyError(f"Missing 'text' on line {line_idx} in {manifest_path}")

            audio_path = str(item["audio_filepath"])
            if not Path(audio_path).exists():
                raise FileNotFoundError(
                    f"Audio file referenced by manifest does not exist on line {line_idx}: {audio_path}"
                )

            item["audio_filepath"] = audio_path
            item["text"] = str(item["text"])
            entries.append(item)

    if not entries:
        raise ValueError(f"Manifest is empty: {manifest_path}")

    return entries


def audit_manifest_entries(name: str, entries: List[Dict[str, object]]) -> None:
    empty_count = 0
    placeholder_count = 0
    word_counts: List[int] = []

    for item in entries:
        text = normalize_text(str(item["text"]))
        if not text:
            empty_count += 1
        if text.lower() in PLACEHOLDER_TOKENS:
            placeholder_count += 1
        word_counts.append(len(text.split()))

    avg_words = sum(word_counts) / max(len(word_counts), 1)
    print(
        f"[info] {name}: {len(entries)} samples | empty_text={empty_count} | "
        f"placeholder_text={placeholder_count} | avg_words={avg_words:.2f}"
    )

    if FAIL_ON_PLACEHOLDER_TRANSCRIPTS and placeholder_count > 0:
        raise ValueError(
            f"{name} contains {placeholder_count} placeholder transcripts. "
            "Fix the manifest text values before training or evaluation."
        )

    if empty_count > 0:
        raise ValueError(f"{name} contains {empty_count} empty transcripts. Fix the manifest before training.")


def levenshtein_distance(ref: Sequence[str], hyp: Sequence[str]) -> int:
    if ref == hyp:
        return 0
    if not ref:
        return len(hyp)
    if not hyp:
        return len(ref)

    previous = list(range(len(hyp) + 1))
    for i, ref_token in enumerate(ref, start=1):
        current = [i]
        for j, hyp_token in enumerate(hyp, start=1):
            substitution_cost = 0 if ref_token == hyp_token else 1
            current.append(
                min(
                    previous[j] + 1,
                    current[j - 1] + 1,
                    previous[j - 1] + substitution_cost,
                )
            )
        previous = current
    return previous[-1]


def compute_wer_cer(references: Sequence[str], hypotheses: Sequence[str]) -> Dict[str, float]:
    if len(references) != len(hypotheses):
        raise ValueError("References and hypotheses must have the same length.")

    word_errors = 0
    word_total = 0
    char_errors = 0
    char_total = 0

    for reference, hypothesis in zip(references, hypotheses):
        ref_text = normalize_text(reference)
        hyp_text = normalize_text(hypothesis)

        ref_words = ref_text.split()
        hyp_words = hyp_text.split()
        word_errors += levenshtein_distance(ref_words, hyp_words)
        word_total += max(len(ref_words), 1)

        ref_chars = list(ref_text)
        hyp_chars = list(hyp_text)
        char_errors += levenshtein_distance(ref_chars, hyp_chars)
        char_total += max(len(ref_chars), 1)

    return {
        "wer": word_errors / max(word_total, 1),
        "cer": char_errors / max(char_total, 1),
    }


def chunked(items: Sequence[str], size: int) -> Iterable[Sequence[str]]:
    for start in range(0, len(items), size):
        yield items[start : start + size]


def make_output_dirs() -> Dict[str, Path]:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path(OUTPUT_ROOT) / f"{RUN_NAME}_{timestamp}"
    output_dirs = {
        "run": run_dir,
        "logs": run_dir / "logs",
        "checkpoints": run_dir / "checkpoints",
        "plots": run_dir / "plots",
        "exports": run_dir / "exports",
        "metrics": run_dir / "metrics",
    }
    for path in output_dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return output_dirs


def write_json(path: Path, payload: Dict[str, object]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def write_rows(path: Path, fieldnames: Sequence[str], rows: Sequence[Dict[str, float]]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def last_finite(rows: Sequence[Dict[str, float]], key: str) -> float:
    for row in reversed(rows):
        value = float(row.get(key, float("nan")))
        if math.isfinite(value):
            return value
    return float("nan")


def save_curve(x_values: Sequence[float], y_values: Sequence[float], title: str, ylabel: str, path: Path) -> None:
    filtered = [(x, y) for x, y in zip(x_values, y_values) if math.isfinite(float(y))]
    if not filtered:
        return
    xs, ys = zip(*filtered)
    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def infer_adapter_in_features(model) -> int:
    candidates = [
        lambda m: int(m.cfg.encoder.d_model),
        lambda m: int(m.cfg.encoder.feat_out),
        lambda m: int(m.cfg.model_defaults.enc_hidden),
        lambda m: int(m.encoder._feat_out),
        lambda m: int(m.decoder.feat_in),
    ]
    for getter in candidates:
        try:
            value = getter(model)
            if value > 0:
                print(f"[info] Adapter input size: {value}")
                return value
        except Exception:
            continue
    raise RuntimeError("Could not infer adapter input size from the restored model.")


def resolve_adapter_name(model, adapter_name: str, preferred_module: str = "encoder") -> str:
    module_names = list(getattr(model, "adapter_module_names", []) or [])
    print(f"[info] Adapter-capable modules: {module_names}")
    if preferred_module and preferred_module in module_names:
        return f"{preferred_module}:{adapter_name}"
    for module_name in module_names:
        if module_name:
            return f"{module_name}:{adapter_name}"
    return adapter_name


def transcribe_manifest(model, manifest_entries: List[Dict[str, object]]) -> List[str]:
    audio_paths = [str(item["audio_filepath"]) for item in manifest_entries]
    predictions: List[str] = []
    was_training = model.training
    model.eval()
    with torch.no_grad():
        for batch_paths in chunked(audio_paths, EVAL_BATCH_SIZE):
            batch_predictions = model.transcribe(
                list(batch_paths),
                batch_size=EVAL_BATCH_SIZE,
                num_workers=EVAL_NUM_WORKERS,
                return_hypotheses=False,
                verbose=False,
            )
            predictions.extend(str(item) for item in batch_predictions)
    if was_training:
        model.train()
    return predictions


def load_lightning_checkpoint_into_model(model, ckpt_path: str) -> None:
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    state_dict = checkpoint.get("state_dict", checkpoint)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"[warn] Missing keys when loading checkpoint: {len(missing)}")
    if unexpected:
        print(f"[warn] Unexpected keys when loading checkpoint: {len(unexpected)}")


class MetricsTrackerCallback(Callback):
    fieldnames = ["epoch", "train_loss", "val_loss", "val_wer", "val_cer", "val_wer_offline"]

    def __init__(self, history_csv_path: Path, val_entries: List[Dict[str, object]]) -> None:
        super().__init__()
        self.history_csv_path = history_csv_path
        self.val_entries = val_entries
        self.history: List[Dict[str, float]] = []
        self.current_train_losses: List[float] = []

    def on_train_epoch_start(self, trainer, pl_module) -> None:
        self.current_train_losses = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx) -> None:
        loss = None
        if isinstance(outputs, dict):
            loss = outputs.get("loss")
        elif hasattr(outputs, "detach"):
            loss = outputs
        if loss is not None:
            self.current_train_losses.append(metric_to_float(loss))

    def on_validation_epoch_end(self, trainer, pl_module) -> None:
        if trainer.sanity_checking:
            return

        metrics = trainer.callback_metrics
        row = {
            "epoch": float(trainer.current_epoch),
            "train_loss": float(mean(self.current_train_losses)) if self.current_train_losses else float("nan"),
            "val_loss": metric_to_float(metrics.get("val_loss")),
            "val_wer": metric_to_float(metrics.get("val_wer")),
            "val_cer": float("nan"),
            "val_wer_offline": float("nan"),
        }

        should_compute_cer = COMPUTE_VAL_CER_EVERY_N_EPOCHS > 0 and ((trainer.current_epoch + 1) % COMPUTE_VAL_CER_EVERY_N_EPOCHS == 0)
        if should_compute_cer:
            predictions = transcribe_manifest(pl_module, self.val_entries)
            references = [str(item["text"]) for item in self.val_entries]
            scores = compute_wer_cer(references, predictions)
            row["val_cer"] = float(scores["cer"])
            row["val_wer_offline"] = float(scores["wer"])
            pl_module.log("val_cer", row["val_cer"], prog_bar=False, logger=True)

        self.history.append(row)
        write_rows(self.history_csv_path, self.fieldnames, self.history)


[info] GPU detected: NVIDIA A100-SXM4-80GB
[info] NumPy version: 1.26.4
[info] CUDA version: 12.8


## Training and Evaluation

The next cell restores the model, enables adapter-based PEFT, trains, evaluates, saves checkpoints, writes metrics, and exports the best model back to Google Drive.


In [ ]:
#@title 4. Define the Training Run

def build_data_config(manifest_path: str, batch_size: int, shuffle: bool) -> Dict[str, object]:
    config = {
        "manifest_filepath": manifest_path,
        "sample_rate": SAMPLE_RATE,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "trim_silence": TRIM_SILENCE,
        "drop_last": False,
    }
    if MIN_DURATION is not None:
        config["min_duration"] = MIN_DURATION
    if MAX_DURATION is not None:
        config["max_duration"] = MAX_DURATION
    return config


def run_experiment() -> Dict[str, object]:
    seed_python(SEED)
    seed_everything(SEED, workers=True)

    for path, description, required in [
        (BASE_MODEL_NEMO_PATH, "Base .nemo model", True),
        (TRAIN_MANIFEST, "Training manifest", True),
        (VAL_MANIFEST, "Validation manifest", True),
        (TEST_MANIFEST, "Test manifest", True),
        (CONTINUE_FROM_FINETUNED_NEMO_PATH, "Continued-training .nemo model", False),
        (CONTINUE_FROM_ADAPTERS_PT_PATH, "Continued-training adapters .pt file", False),
        (RESUME_FROM_CHECKPOINT_PATH, "Resume checkpoint", False),
    ]:
        ensure_exists(path, description, required=required)

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision(MATMUL_PRECISION)

    trainer_deterministic = "warn" if DETERMINISTIC else False
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = False
        print("[info] Deterministic mode is set to warn-only because GPU CTC backward is not strictly deterministic.")

    output_dirs = make_output_dirs()
    print(f"[info] Run directory: {output_dirs['run']}")

    train_entries = load_manifest(TRAIN_MANIFEST)
    val_entries = load_manifest(VAL_MANIFEST)
    test_entries = load_manifest(TEST_MANIFEST)
    audit_manifest_entries("train", train_entries)
    audit_manifest_entries("val", val_entries)
    audit_manifest_entries("test", test_entries)

    restore_nemo_path = CONTINUE_FROM_FINETUNED_NEMO_PATH or BASE_MODEL_NEMO_PATH
    logger = CSVLogger(save_dir=str(output_dirs["logs"]), name="lightning")

    history_callback = MetricsTrackerCallback(output_dirs["metrics"] / "history.csv", val_entries)
    best_checkpoint = ModelCheckpoint(
        dirpath=str(output_dirs["checkpoints"]),
        filename="best-epoch={epoch:02d}-val_wer={val_wer:.4f}",
        monitor="val_wer",
        mode="min",
        save_top_k=SAVE_TOP_K,
        save_last=True,
        auto_insert_metric_name=False,
    )
    periodic_checkpoint = ModelCheckpoint(
        dirpath=str(output_dirs["checkpoints"]),
        filename="epoch={epoch:02d}",
        every_n_epochs=PERIODIC_CHECKPOINT_EVERY_N_EPOCHS,
        save_top_k=-1,
        auto_insert_metric_name=False,
    )

    callbacks = [
        history_callback,
        best_checkpoint,
        periodic_checkpoint,
        LearningRateMonitor(logging_interval="step"),
    ]
    if ENABLE_EARLY_STOPPING:
        callbacks.append(
            EarlyStopping(
                monitor=EARLY_STOPPING_MONITOR,
                mode=EARLY_STOPPING_MODE,
                min_delta=EARLY_STOPPING_MIN_DELTA,
                patience=EARLY_STOPPING_PATIENCE,
                verbose=True,
            )
        )

    trainer = Trainer(
        accelerator="gpu",
        devices=1,
        precision=PRECISION,
        logger=logger,
        callbacks=callbacks,
        max_epochs=MAX_EPOCHS,
        accumulate_grad_batches=GRAD_ACCUMULATION_STEPS,
        gradient_clip_val=GRAD_CLIP_VAL,
        deterministic=trainer_deterministic,
        log_every_n_steps=LOG_EVERY_N_STEPS,
        num_sanity_val_steps=NUM_SANITY_VAL_STEPS,
        check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
        limit_train_batches=LIMIT_TRAIN_BATCHES,
        limit_val_batches=LIMIT_VAL_BATCHES,
        limit_test_batches=LIMIT_TEST_BATCHES,
        default_root_dir=str(output_dirs["run"]),
    )

    print(f"[info] Restoring model from: {restore_nemo_path}")
    model = nemo_asr.models.ASRModel.restore_from(
        restore_path=restore_nemo_path,
        trainer=trainer,
        map_location=DEVICE,
    )

    if "CTC" not in model.__class__.__name__:
        raise ValueError("Please provide a FastConformer-CTC .nemo checkpoint.")

    if hasattr(model, "change_subsampling_conv_chunking_factor"):
        model.change_subsampling_conv_chunking_factor(SUBSAMPLING_CONV_CHUNKING_FACTOR)
    elif hasattr(model, "encoder") and hasattr(model.encoder, "change_subsampling_conv_chunking_factor"):
        model.encoder.change_subsampling_conv_chunking_factor(SUBSAMPLING_CONV_CHUNKING_FACTOR)

    with open_dict(model.cfg):
        model.cfg.train_ds = OmegaConf.create(build_data_config(TRAIN_MANIFEST, TRAIN_BATCH_SIZE, True))
        model.cfg.validation_ds = OmegaConf.create(build_data_config(VAL_MANIFEST, VAL_BATCH_SIZE, False))
        model.cfg.test_ds = OmegaConf.create(build_data_config(TEST_MANIFEST, TEST_BATCH_SIZE, False))
        model.cfg.use_cer = False
        model.cfg.log_prediction = not DISABLE_NEMO_PREDICTION_LOGS
        model.cfg.optim = OmegaConf.create(
            {
                "name": "adamw",
                "lr": LEARNING_RATE,
                "betas": list(BETAS),
                "weight_decay": WEIGHT_DECAY,
                "sched": {
                    "name": "CosineAnnealing",
                    "warmup_ratio": WARMUP_RATIO,
                    "min_lr": MIN_LR,
                    "max_steps": -1,
                    "monitor": "val_wer",
                    "reduce_on_plateau": False,
                },
            }
        )
        model.cfg.spec_augment = OmegaConf.create(
            {
                "_target_": "nemo.collections.asr.modules.SpectrogramAugmentation",
                "freq_masks": SPEC_AUG_FREQ_MASKS,
                "freq_width": SPEC_AUG_FREQ_WIDTH,
                "time_masks": SPEC_AUG_TIME_MASKS,
                "time_width": SPEC_AUG_TIME_WIDTH,
            }
        ) if ENABLE_SPEC_AUGMENT else None

    model.spec_augmentation = model.from_config_dict(model.cfg.spec_augment) if ENABLE_SPEC_AUGMENT else None
    if DISABLE_NEMO_PREDICTION_LOGS:
        if hasattr(model, "log_prediction"):
            model.log_prediction = False
        if hasattr(model, "wer") and hasattr(model.wer, "log_prediction"):
            model.wer.log_prediction = False

    if USE_PEFT:
        print("[info] Enabling NeMo adapter-based PEFT.")
        model.replace_adapter_compatible_modules(update_config=True, verbose=True)
        requested_adapter_name = resolve_adapter_name(model, ADAPTER_NAME, PEFT_TARGET_MODULE)

        if CONTINUE_FROM_ADAPTERS_PT_PATH:
            print(f"[info] Loading adapters from: {CONTINUE_FROM_ADAPTERS_PT_PATH}")
            model.load_adapters(CONTINUE_FROM_ADAPTERS_PT_PATH)
            enabled = model.get_enabled_adapters() if model.is_adapter_available() else []
            if not enabled:
                raise RuntimeError("Adapter weights were loaded, but no adapter was enabled.")
            active_adapter_name = enabled[0]
        elif not model.is_adapter_available():
            adapter_cfg = adapter_modules.LinearAdapterConfig(
                in_features=infer_adapter_in_features(model),
                dim=ADAPTER_DIM,
                dropout=ADAPTER_DROPOUT,
                activation=ADAPTER_ACTIVATION,
                norm_position=ADAPTER_NORM_POSITION,
            )
            model.add_adapter(name=requested_adapter_name, cfg=adapter_cfg)
            active_adapter_name = requested_adapter_name
        else:
            enabled = model.get_enabled_adapters()
            active_adapter_name = enabled[0] if enabled else requested_adapter_name

        model.set_enabled_adapters(enabled=False)
        model.set_enabled_adapters(name=active_adapter_name, enabled=True)
        model.freeze()
        model.unfreeze_enabled_adapters()
        print(f"[info] Active adapter: {active_adapter_name}")
    else:
        print("[info] PEFT disabled; full fine-tuning will be used.")

    model.setup_training_data(model.cfg.train_ds)
    model.setup_validation_data(model.cfg.validation_ds)
    model.setup_test_data(model.cfg.test_ds)

    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    trainable_pct = 100.0 * trainable_params / max(total_params, 1)
    print(f"[info] Total params: {total_params:,}")
    print(f"[info] Trainable params: {trainable_params:,} ({trainable_pct:.4f}%)")

    write_json(
        output_dirs["metrics"] / "run_config.json",
        {
            "base_model_nemo_path": BASE_MODEL_NEMO_PATH,
            "continue_from_finetuned_nemo_path": CONTINUE_FROM_FINETUNED_NEMO_PATH,
            "continue_from_adapters_pt_path": CONTINUE_FROM_ADAPTERS_PT_PATH,
            "resume_from_checkpoint_path": RESUME_FROM_CHECKPOINT_PATH,
            "train_manifest": TRAIN_MANIFEST,
            "val_manifest": VAL_MANIFEST,
            "test_manifest": TEST_MANIFEST,
            "use_peft": USE_PEFT,
            "adapter_dim": ADAPTER_DIM,
            "max_epochs": MAX_EPOCHS,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "grad_accumulation_steps": GRAD_ACCUMULATION_STEPS,
            "learning_rate": LEARNING_RATE,
            "enable_early_stopping": ENABLE_EARLY_STOPPING,
            "early_stopping_monitor": EARLY_STOPPING_MONITOR,
            "early_stopping_mode": EARLY_STOPPING_MODE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "precision": PRECISION,
            "seed": SEED,
            "total_params": total_params,
            "trainable_params": trainable_params,
            "trainable_pct": trainable_pct,
        },
    )

    print("[info] Starting training...")
    trainer.fit(model, ckpt_path=RESUME_FROM_CHECKPOINT_PATH)

    best_model_path = best_checkpoint.best_model_path or ""
    if best_model_path:
        print(f"[info] Loading best checkpoint for export and evaluation: {best_model_path}")
        load_lightning_checkpoint_into_model(model, best_model_path)
    else:
        print("[warn] No best checkpoint path was found; using the final in-memory weights.")

    test_results = trainer.test(model)

    print("[info] Computing final validation metrics...")
    val_predictions = transcribe_manifest(model, val_entries)
    val_scores = compute_wer_cer([str(item["text"]) for item in val_entries], val_predictions)

    print("[info] Computing final test metrics...")
    test_predictions = transcribe_manifest(model, test_entries)
    test_scores = compute_wer_cer([str(item["text"]) for item in test_entries], test_predictions)

    history = history_callback.history
    summary = {
        "train_loss_last": last_finite(history, "train_loss"),
        "val_loss_last": last_finite(history, "val_loss"),
        "val_wer_last": last_finite(history, "val_wer"),
        "val_cer_last": last_finite(history, "val_cer"),
        "val_wer_offline_final": float(val_scores["wer"]),
        "val_cer_offline_final": float(val_scores["cer"]),
        "test_wer_final": float(test_scores["wer"]),
        "test_cer_final": float(test_scores["cer"]),
        "lightning_test_metrics": test_results[0] if test_results else {},
        "best_model_checkpoint": best_model_path,
        "last_checkpoint": best_checkpoint.last_model_path,
        "run_dir": str(output_dirs["run"]),
    }
    write_json(output_dirs["metrics"] / "summary.json", summary)

    predictions_path = output_dirs["metrics"] / "test_predictions.jsonl"
    with open(predictions_path, "w", encoding="utf-8") as f:
        for item, prediction in zip(test_entries, test_predictions):
            f.write(
                json.dumps(
                    {
                        "audio_filepath": item["audio_filepath"],
                        "reference": item["text"],
                        "prediction": prediction,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

    best_nemo_path = output_dirs["exports"] / "best_model.nemo"
    final_nemo_path = output_dirs["exports"] / "final_model.nemo"
    adapters_pt_path = output_dirs["exports"] / "best_adapters.pt"

    for export_path in [best_nemo_path, final_nemo_path, adapters_pt_path]:
        if export_path.exists() and OVERWRITE_EXISTING_EXPORTS:
            export_path.unlink()

    print(f"[info] Exporting best model to: {best_nemo_path}")
    model.save_to(str(best_nemo_path))
    shutil.copy2(best_nemo_path, final_nemo_path)

    if USE_PEFT and model.is_adapter_available():
        print(f"[info] Exporting adapters to: {adapters_pt_path}")
        model.save_adapters(str(adapters_pt_path))

    epochs = [row["epoch"] for row in history]
    save_curve(epochs, [row["train_loss"] for row in history], "Train Loss", "Loss", output_dirs["plots"] / "train_loss_curve.png")
    save_curve(epochs, [row["val_loss"] for row in history], "Validation Loss", "Loss", output_dirs["plots"] / "val_loss_curve.png")
    save_curve(epochs, [row["val_wer"] for row in history], "Validation WER", "WER", output_dirs["plots"] / "val_wer_curve.png")
    save_curve(epochs, [row["val_cer"] for row in history], "Validation CER", "CER", output_dirs["plots"] / "val_cer_curve.png")

    print("[info] Training complete.")
    print(json.dumps(summary, indent=2))
    return summary


In [ ]:
#@title 5. Run Training

# Re-run this cell after editing the config cell above.
summary = run_experiment()
summary


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


[info] Deterministic mode is set to warn-only because GPU CTC backward is not strictly deterministic.
[info] Run directory: /content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_runs/fastconformer_ctc_small_peft_20260321_044750


INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..


[info] train: 4859 samples | empty_text=0 | placeholder_text=0 | avg_words=35.26
[info] val: 571 samples | empty_text=0 | placeholder_text=0 | avg_words=35.26
[info] test: 287 samples | empty_text=0 | placeholder_text=0 | avg_words=35.10
[info] Restoring model from: /content/drive/MyDrive/shaswot_mate/angrezi/fastconformer.nemo
[NeMo I 2026-03-21 04:47:52 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-21 04:47:52 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-03-21 04:47:52 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librisp

...

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        global_step        │          1824.0           │
│         test_loss         │     9.140547752380371     │
│         test_wer          │    0.04486352205276489    │
└───────────────────────────┴───────────────────────────┘

[info] Computing final validation metrics...
[info] Computing final test metrics...
[info] Exporting best model to: /content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_runs/fastconformer_ctc_small_peft_20260321_044750/exports/best_model.nemo
[info] Exporting adapters to: /content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_runs/fastconformer_ctc_small_peft_20260321_044750/exports/best_adapters.pt
[info] Training complete.
{
  "train_loss_last": 8.538994942840777,
  "val_loss_last": 5.630991458892822,
  "val_wer_last": 0.03298231586813927,
  "val_cer_last": 11.198901098901098,
  "val_wer_offline_final": 9.48643949930459,
  "val_cer_offline_final": 11.19844477556342,
  "test_wer_final": 9.363546113372381,
  "test_cer_final": 11.06051980428271,
  "lightning_test_metrics": {
    "global_step": 1824.0,
    "test_loss": 9.140547752380371,
    "test_wer": 0.04486352205276489
  },
  "best_model_checkpoint": "/content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_ru

{'train_loss_last': 8.538994942840777,
 'val_loss_last': 5.630991458892822,
 'val_wer_last': 0.03298231586813927,
 'val_cer_last': 11.198901098901098,
 'val_wer_offline_final': 9.48643949930459,
 'val_cer_offline_final': 11.19844477556342,
 'test_wer_final': 9.363546113372381,
 'test_cer_final': 11.06051980428271,
 'lightning_test_metrics': {'global_step': 1824.0,
  'test_loss': 9.140547752380371,
  'test_wer': 0.04486352205276489},
 'best_model_checkpoint': '/content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_runs/fastconformer_ctc_small_peft_20260321_044750/checkpoints/best-epoch=07-val_wer=0.0329.ckpt',
 'last_checkpoint': '/content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_runs/fastconformer_ctc_small_peft_20260321_044750/checkpoints/last.ckpt',
 'run_dir': '/content/drive/MyDrive/shaswot_mate/angrezi/nemo_fastconformer_runs/fastconformer_ctc_small_peft_20260321_044750'}